![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

In [95]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler


# Import any additional modules and start coding below
df_rental = pd.read_csv("rental_info.csv")

In [96]:
df_rental.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   rental_date       15861 non-null  object 
 1   return_date       15861 non-null  object 
 2   amount            15861 non-null  float64
 3   release_year      15861 non-null  float64
 4   rental_rate       15861 non-null  float64
 5   length            15861 non-null  float64
 6   replacement_cost  15861 non-null  float64
 7   special_features  15861 non-null  object 
 8   NC-17             15861 non-null  int64  
 9   PG                15861 non-null  int64  
 10  PG-13             15861 non-null  int64  
 11  R                 15861 non-null  int64  
 12  amount_2          15861 non-null  float64
 13  length_2          15861 non-null  float64
 14  rental_rate_2     15861 non-null  float64
dtypes: float64(8), int64(4), object(3)
memory usage: 1.8+ MB


In [97]:
df_rental["rental_length"] = pd.to_datetime(df_rental["return_date"]) - pd.to_datetime(df_rental["rental_date"])
df_rental["rental_length_days"] = df_rental["rental_length"].dt.days


In [98]:
# Create dummy columns for 'special_features'
df_rental['deleted_scenes'] = df_rental['special_features'].str.contains('Deleted Scenes').astype(int)
df_rental['behind_the_scenes'] = df_rental['special_features'].str.contains('Behind the Scenes').astype(int)


In [99]:
df_rental["deleted_scenes"] =  np.where(df_rental["special_features"].str.contains("Deleted Scenes"), 1, 0)
df_rental["behind_the_scenes"] =  np.where(df_rental["special_features"].str.contains("Behind the Scenes"), 1, 0)



In [100]:
cols_to_drop = ["special_features", "rental_length", "rental_length_days", "rental_date", "return_date"]


In [101]:
X = df_rental.drop(cols_to_drop, axis=1)
y = df_rental["rental_length_days"]


In [102]:
X_train,X_test,y_train,y_test = train_test_split(X, 
                                                 y, 
                                                 test_size=0.2, 
                                                 random_state=9)

In [103]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.3, random_state=9) 

lasso.fit(X_train, y_train)
lasso_coef = lasso.coef_

X_lasso_train, X_lasso_test = X_train.iloc[:, lasso_coef > 0], X_test.iloc[:, lasso_coef > 0]


In [104]:
from sklearn.linear_model import LinearRegression
Lin_reg = LinearRegression()
Lin_reg = Lin_reg.fit(X_lasso_train, y_train)
y_test_pred = Lin_reg.predict(X_lasso_test)
mse_lin_reg_lasso = mean_squared_error(y_test, y_test_pred)
print(f"MSE de Regresion Linear {mse_lin_reg_lasso}.")




MSE de Regresion Linear 4.812297241276244.


In [105]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestRegressor()
rf.fit(X_lasso_train, y_train)
y_test_pred = rf.predict(X_lasso_test)
mse_for_rf = mean_squared_error(y_test, y_test_pred)
print(f"MSE de RandomForest {mse_for_rf}.")

MSE de RandomForest 3.628422532782255.


In [106]:
params = {
    'n_estimators': np.arange(1, 50, 1),
    'max_depth': np.arange(1, 20, 1)
}
rf = RandomForestRegressor()
random_search_cv = RandomizedSearchCV(rf, params, cv=5, random_state=9)
random_search_cv.fit(X_train, y_train)
hyper_parameters = random_search_cv.best_params_

rf_final = RandomForestRegressor(**hyper_parameters, random_state=9)
rf_final.fit(X_train, y_train)
y_pred = rf_final.predict(X_test)
mse_for_rf_hyper_prameters = mean_squared_error(y_test, y_pred)
print(f"MSE de RandomForest con HyperParametros {mse_for_rf_hyper_prameters}.")

MSE de RandomForest con HyperParametros 2.0252434990290515.


In [107]:
best_model = rf_final
best_mse = mse_for_rf_hyper_prameters
print(f"El mejor modeloo es {best_model}.")
print(f"Tiene un MSE de  {mse_for_rf_hyper_prameters}.")

El mejor modeloo es RandomForestRegressor(max_depth=16, n_estimators=33, random_state=9).
Tiene un MSE de  2.0252434990290515.
